In [ ]:
!pip install pandas matplotlib seaborn wordcloud nltk openai scikit-learn gensim spacy langdetect tiktoken

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from textblob import TextBlob
import openai

nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load data
posts = pd.read_csv(r"C:\Users\ysire\PycharmProjects\PythonProject\gmat_posts.csv")
comments = pd.read_csv(r"C:\Users\ysire\PycharmProjects\PythonProject\gmat_comments.csv")

print("Loaded", len(posts), "posts and", len(comments), "comments")


In [ ]:

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'\@\w+|\#','', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(filtered)

comments['clean_body'] = comments['body'].apply(clean_text)


In [ ]:

all_words = ' '.join(comments['clean_body'])
word_freq = Counter(all_words.split())
common_words = word_freq.most_common(20)

# Bar chart
pd.DataFrame(common_words, columns=['word', 'freq']).set_index('word').plot(kind='bar', legend=False, title="Top 20 Words")
plt.show()

# Word Cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_words)
plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud of GMAT Conversations")
plt.show()


In [ ]:

comments['sentiment'] = comments['clean_body'].apply(lambda text: TextBlob(text).sentiment.polarity)
comments['sentiment_label'] = comments['sentiment'].apply(lambda x: 'positive' if x > 0.1 else 'negative' if x < -0.1 else 'neutral')
comments['sentiment_label'].value_counts().plot(kind='bar', title="Sentiment Distribution")
plt.show()


In [ ]:

section_keywords = {
    'quant': ['quant', 'math', 'algebra', 'geometry', 'arithmetic', 'data sufficiency'],
    'verbal': ['verbal', 'rc', 'reading comprehension', 'sentence correction', 'sc', 'cr', 'critical reasoning'],
    'data_insights': ['data insights', 'di section', 'charts', 'tables', 'data interpretation', 'graphs', 'multi-source', 'di question']
}

for section, keywords in section_keywords.items():
    comments[section] = comments['clean_body'].apply(lambda x: 1 if any(kw in x for kw in keywords) else 0)

section_counts = comments[['quant', 'verbal', 'data_insights']].sum().sort_values(ascending=False)
section_counts.plot(kind='bar', title="Mentioned GMAT Sections")
plt.show()


In [ ]:

from transformers import pipeline

# Load summarization model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_with_hf(texts, topic_hint=None):
    joined_text = " ".join(texts[:20])
    if len(joined_text) > 4000:
        joined_text = joined_text[:4000]  # Limit input for performance
    summary = summarizer(joined_text, max_length=120, min_length=30, do_sample=False)
    print("🔍", topic_hint.upper() if topic_hint else "Summary", "Summary:")
    print(summary[0]['summary_text'])

# Example: Summarize Data Insights section
di_comments = comments[comments['data_insights'] == 1].sample(n=30, random_state=42)
summarize_with_hf(di_comments['clean_body'].tolist(), topic_hint="Data Insights section")
